# ACES to sRGB Conversion & Image Degradation Exploration

This notebook explores the synthesis of training data for LuminaScale by:
1. Converting ACES2065-1 EXR images to sRGB using OpenImageIO
2. Applying synthetic degradations to simulate 8-bit AI-generated images
3. Visualizing the effects of quantization, color warping, and artifacts

Based on the project plan's "Input Generation" strategy.

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import PowerNorm
import OpenImageIO as oiio
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import gaussian_blur
from scipy.ndimage import gaussian_filter
from scipy.signal import medfilt2d
import warnings

warnings.filterwarnings("ignore")

# Project paths - navigate up from notebooks to project root
project_root = Path.cwd().parent
aces_dir = project_root / "dataset" / "temp" / "aces"
output_dir = project_root / "dataset" / "temp" / "srgb_degraded"
output_dir.mkdir(exist_ok=True, parents=True)

# Get list of ACES EXR files
aces_files = sorted(list(aces_dir.glob("*.exr")))
if not aces_files:
    # Try alternative pattern
    aces_files = sorted(list(aces_dir.glob("*_aces.exr")))
print(f"Found {len(aces_files)} ACES images")
print(f"Sample files: {[f.name for f in aces_files[:3]]}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Section 1: Load and Convert ACES Images to sRGB

Load 16-bit ACES2065-1 EXR images and convert them to sRGB color space using OpenImageIO.
ACES2065-1 is the reference color space (linear, wide gamut); sRGB is the target display space (gamma-corrected, narrow gamut).

In [ ]:
def load_aces_and_convert_srgb(
    aces_path: Path | str, target_colorspace: str = "sRGB"
) -> np.ndarray:
    """Load ACES2065-1 EXR and convert to target color space using OIIO.
    
    Args:
        aces_path: Path to ACES EXR file.
        target_colorspace: Target color space (default: "sRGB").
    
    Returns:
        Image as float32 numpy array in [0, 1] range or beyond for HDR.
    """
    # Open the EXR file
    inp = oiio.ImageInput.open(str(aces_path))
    assert inp, f"Failed to open {aces_path}"
    
    # Read image as float
    image_spec = inp.spec()
    image = inp.read_image()
    inp.close()
    
    assert image is not None, f"Failed to read image from {aces_path}"
    
    # Convert ACES2065-1 -> target color space using OIIO's color conversion
    # Note: OIIO's Python API requires positional or keyword arguments that match the overload.
    # We use the raw image array directly for initialization.
    buf = oiio.ImageBuf()
    buf.set_pixels(oiio.ROI(), image)
    
    # Use the simpler colorconvert that returns converted ImageBuf
    converted_buf = oiio.ImageBufAlgo.colorconvert(
        buf, "ACES2065-1", target_colorspace
    )
    
    # Extract numpy array from ImageBuf
    converted = np.array(converted_buf.get_pixels())
    h = converted_buf.spec().height
    w = converted_buf.spec().width
    c = converted_buf.spec().nchannels
    converted = converted.reshape(h, w, c)
    
    # Ensure float32 and handle potential NaN/Inf
    converted = converted.astype(np.float32)
    converted = np.nan_to_num(converted, nan=0.0, posinf=1.0, neginf=0.0)
    
    return converted

In [ ]:
def normalize_for_display(img: np.ndarray, percentile_clip: float = 98.0) -> np.ndarray:
    """Normalize image to [0, 1] for display, handling HDR values gracefully."""
    img_clipped = np.percentile(img, percentile_clip)
    normalized = np.clip(img / img_clipped, 0, 1)
    return normalized.astype(np.float32)


# Visualize first few ACES images
if aces_files and len(aces_files) >= 1:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, (ax, aces_path) in enumerate(zip(axes, aces_files[:3])):
        aces_image = load_aces_and_convert_srgb(aces_path, "sRGB")
        # Normalize for display
        display_image = normalize_for_display(aces_image)
        
        # Handle both RGB and RGBA
        if display_image.shape[2] == 4:
            display_image = display_image[:, :, :3]
        
        ax.imshow(display_image)
        ax.set_title(f"{aces_path.name}\n(sRGB converted)")
        ax.axis("off")
    
    plt.tight_layout()
    plt.savefig(output_dir / "01_original_srgb.png", dpi=100, bbox_inches="tight")
    plt.title("Original ACES Images Converted to sRGB")
    plt.show()
    print("✓ Saved visualization: 01_original_srgb.png")

## Section 2: Apply Arbitrary Color Warping

Simulate extreme color shifts and saturation changes by applying a random 3×3 color transformation matrix.
This simulates the kind of stylization or color grading artifacts found in AI-generated images.

In [ ]:
def apply_color_warping(
    img: np.ndarray, min_coeff: float = -0.5, max_coeff: float = 2.5, seed: int | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Apply a random 3x3 color transformation matrix.
    
    Args:
        img: Input image, shape [H, W, C].
        min_coeff: Minimum coefficient value.
        max_coeff: Maximum coefficient value.
        seed: Random seed for reproducibility.
    
    Returns:
        Tuple of (warped_image, color_matrix).
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate random 3x3 color transformation matrix
    color_matrix = np.random.uniform(min_coeff, max_coeff, size=(3, 3))
    
    # Reshape image to [H*W, C], apply matrix, reshape back
    h, w, c = img.shape
    if c > 3:
        # Handle RGBA by processing only RGB channels
        rgb = img[:, :, :3]
        alpha = img[:, :, 3:]
    else:
        rgb = img
        alpha = None
    
    img_flat = rgb.reshape(-1, 3)
    warped_flat = img_flat @ color_matrix.T
    warped = warped_flat.reshape(h, w, 3)
    
    if alpha is not None:
        warped = np.concatenate([warped, alpha], axis=-1)
    
    return warped.astype(np.float32), color_matrix


# Test color warping on first image
if aces_files:
    test_image = load_aces_and_convert_srgb(aces_files[0], "sRGB")
    warped_image, matrix = apply_color_warping(test_image, seed=42)
    
    print("Color warping applied:")
    print(f"Color transformation matrix:\n{matrix}")
    print(f"Original image range: [{test_image.min():.4f}, {test_image.max():.4f}]")
    print(f"Warped image range: [{warped_image.min():.4f}, {warped_image.max():.4f}]")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].imshow(normalize_for_display(test_image[:, :, :3]))
    axes[0].set_title("Original (sRGB)")
    axes[0].axis("off")
    
    axes[1].imshow(normalize_for_display(warped_image[:, :, :3]))
    axes[1].set_title("Color Warped")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.savefig(output_dir / "02_color_warping.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("✓ Saved visualization: 02_color_warping.png")

## Section 3: Apply Stylized Contrast Curves

Simulate crushed blacks and blown-out highlights characteristic of AI-generated images using sigmoidal or power-law curves.

In [ ]:
def sigmoid_curve(x: np.ndarray, steepness: float = 5.0, midpoint: float = 0.5) -> np.ndarray:
    """Apply sigmoid contrast curve (S-curve) to enhance midtones."""
    return 1.0 / (1.0 + np.exp(-steepness * (x - midpoint)))


def power_curve(x: np.ndarray, gamma: float = 2.0) -> np.ndarray:
    """Apply power-law (gamma) curve."""
    return np.power(np.clip(x, 0, 1), 1.0 / gamma)


def apply_contrast_curve(
    img: np.ndarray, curve_type: str = "sigmoid", params: dict | None = None, seed: int | None = None
) -> tuple[np.ndarray, dict]:
    """Apply stylized contrast curve to image.
    
    Args:
        img: Input image, shape [H, W, C].
        curve_type: "sigmoid" or "power".
        params: Dictionary with curve parameters.
        seed: Random seed.
    
    Returns:
        Tuple of (image with contrast curve applied, parameter dictionary).
    """
    if seed is not None:
        np.random.seed(seed)
    
    if params is None:
        if curve_type == "sigmoid":
            params = {
                "steepness": np.random.uniform(2.0, 8.0),
                "midpoint": np.random.uniform(0.3, 0.7),
            }
        else:  # power
            params = {"gamma": np.random.uniform(0.5, 3.0)}
    
    # Apply curve
    if curve_type == "sigmoid":
        curved = sigmoid_curve(img, **params)
    else:
        curved = power_curve(img, **params)
    
    return curved.astype(np.float32), params

## Section 4: Simulate AI Artifacts

Add realistic defects found in AI-generated images:
- **Chroma noise**: Low-frequency color blotches (typical of diffusion model color bleeding)
- **Unsharp mask**: Over-sharpening applied on top (common in upscaling)

In [ ]:
def add_chroma_noise(
    img: np.ndarray, noise_strength: float = 0.1, blur_sigma: float = 20.0, seed: int | None = None, device: str = "cuda"
) -> np.ndarray:
    """Add low-frequency chroma noise to simulate diffusion artifacts (GPU-accelerated).
    
    Args:
        img: Input image, shape [H, W, C].
        noise_strength: Amplitude of color noise.
        blur_sigma: Gaussian blur sigma for low-frequency effect.
        seed: Random seed.
        device: Device to run on ("cuda" or "cpu").
    
    Returns:
        Image with chroma noise added.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    
    # Convert to torch tensor [H, W, C] -> [1, C, H, W] for conv operations
    img_t = torch.from_numpy(img).to(device).unsqueeze(0).permute(0, 3, 1, 2).float()
    
    # Generate noise on GPU
    noise = torch.randn_like(img_t) * noise_strength
    
    # Apply Gaussian blur using PyTorch (much faster than scipy)
    kernel_size = int(6 * blur_sigma + 1)
    if kernel_size % 2 == 0:
        kernel_size += 1
    noise_blurred = gaussian_blur(noise, kernel_size=kernel_size, sigma=blur_sigma)
    
    # Add noise and convert back
    result = img_t + noise_blurred
    result = result.permute(0, 2, 3, 1).squeeze(0).cpu().numpy().astype(np.float32)
    
    return result


def apply_unsharp_mask(img: np.ndarray, radius: float = 1.0, strength: float = 2.0, device: str = "cuda") -> np.ndarray:
    """Apply unsharp masking (over-sharpening) using GPU acceleration.
    
    Args:
        img: Input image.
        radius: Blur radius for the mask.
        strength: Sharpening strength (1.0 = subtle, 2.0+ = aggressive).
        device: Device to run on ("cuda" or "cpu").
    
    Returns:
        Over-sharpened image.
    """
    # Convert to torch tensor [H, W, C] -> [1, C, H, W]
    img_t = torch.from_numpy(img).to(device).unsqueeze(0).permute(0, 3, 1, 2).float()
    
    # Apply Gaussian blur using PyTorch
    kernel_size = int(6 * radius + 1)
    if kernel_size % 2 == 0:
        kernel_size += 1
    blurred = gaussian_blur(img_t, kernel_size=kernel_size, sigma=radius)
    
    # Compute high-pass and sharpen
    high_pass = img_t - blurred
    sharpened = img_t + strength * high_pass
    
    # Convert back to numpy
    result = sharpened.permute(0, 2, 3, 1).squeeze(0).cpu().numpy().astype(np.float32)
    
    return result


# Test AI artifact simulation
if aces_files:
    test_image = load_aces_and_convert_srgb(aces_files[0], "sRGB")
    test_image_clipped = np.clip(test_image, 0, 1)
    
    # Apply artifacts (using GPU)
    with_chroma = add_chroma_noise(test_image_clipped, noise_strength=0.05, blur_sigma=20.0, seed=42, device=device)
    with_chroma = np.clip(with_chroma, 0, 1)
    
    with_sharpening = apply_unsharp_mask(with_chroma, radius=1.0, strength=1.5, device=device)
    with_sharpening = np.clip(with_sharpening, 0, 1)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].imshow(test_image_clipped[:, :, :3])
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(with_chroma[:, :, :3])
    axes[1].set_title("+ Chroma Noise")
    axes[1].axis("off")
    
    axes[2].imshow(with_sharpening[:, :, :3])
    axes[2].set_title("+ Unsharp Mask")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.savefig(output_dir / "04_ai_artifacts.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("✓ Saved visualization: 04_ai_artifacts.png")


## Section 5: Apply 8-bit Quantization

The final step: convert smooth floating-point values to discrete 8-bit integers, introducing quantization banding.
This is what the BDE (Bit-Depth Expansion) head must learn to remove.

In [ ]:
def quantize_to_8bit(img: np.ndarray, dither: bool = False, seed: int | None = None) -> np.ndarray:
    """Quantize image to 8-bit without dithering (introduces banding).
    
    Args:
        img: Input image in [0, 1] range.
        dither: If True, apply ordered dithering to reduce banding.
        seed: Random seed for dithering.
    
    Returns:
        Quantized image in [0, 1] range (float32, representing 8-bit values).
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Clip to [0, 1]
    img_clipped = np.clip(img, 0, 1)
    
    if dither:
        # Add small random noise before quantization (ordered dithering)
        dither_noise = np.random.uniform(-0.5 / 256, 0.5 / 256, size=img_clipped.shape)
        img_clipped = img_clipped + dither_noise
    
    # Quantize to 256 levels and back
    quantized = np.round(img_clipped * 255) / 255
    
    return quantized.astype(np.float32)


def compute_banding_energy(img: np.ndarray) -> float:
    """Compute a metric for banding artifacts (gradient histogram peakiness)."""
    # Compute gradients
    grad_x = np.diff(img, axis=1)
    grad_y = np.diff(img, axis=0)
    
    # Flatten and compute histogram
    grads = np.concatenate([grad_x.flat, grad_y.flat])
    grads_clipped = np.clip(np.abs(grads), 0, 0.05)  # Focus on small gradients
    
    # Higher concentration in small gradients = more banding
    hist, _ = np.histogram(grads_clipped, bins=50, range=(0, 0.05))
    energy = np.sum(hist ** 2)  # Peakiness measure
    
    return energy


# Test quantization and banding
if aces_files:
    test_image = load_aces_and_convert_srgb(aces_files[0], "sRGB")
    test_image_clipped = np.clip(test_image, 0, 1)
    
    # Get a smooth region for better visualization
    smooth_region = test_image_clipped[100:300, 100:300, :3]
    
    quantized_no_dither = quantize_to_8bit(smooth_region, dither=False, seed=42)
    quantized_with_dither = quantize_to_8bit(smooth_region, dither=True, seed=42)
    
    # Compute banding energy
    energy_original = compute_banding_energy(smooth_region)
    energy_quantized = compute_banding_energy(quantized_no_dither)
    energy_dithered = compute_banding_energy(quantized_with_dither)
    
    print(f"Banding energy (original): {energy_original:.2e}")
    print(f"Banding energy (quantized): {energy_quantized:.2e}")
    print(f"Banding energy (dithered): {energy_dithered:.2e}")
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].imshow(smooth_region)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(quantized_no_dither)
    axes[1].set_title("8-bit Quantized\n(Banding visible)")
    axes[1].axis("off")
    
    axes[2].imshow(quantized_with_dither)
    axes[2].set_title("8-bit with Dithering\n(Noise reduces banding)")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.savefig(output_dir / "05_quantization.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("✓ Saved visualization: 05_quantization.png")

## Section 6: Visualize Full Degradation Pipeline

Combine all degradation steps (color warp → contrast curve → artifacts → quantization) and visualize the progression.
This represents the synthetic "AI-generated" input that the model must restore.

In [ ]:
def apply_full_degradation(
    img: np.ndarray, seed: int | None = None, device: str = "cuda"
) -> dict[str, np.ndarray]:
    """Apply complete degradation pipeline (color warp → contrast → artifacts → quantize).
    
    Returns a dictionary with intermediate and final results.
    """
    if seed is not None:
        np.random.seed(seed)
    
    results = {"original": img.copy()}
    
    # Step 1: Color warping
    warped, _ = apply_color_warping(img, seed=seed)
    results["3_color_warped"] = np.clip(warped, 0, 1)
    
    # Step 2: Contrast curve
    clipped = np.clip(results["3_color_warped"], 0, 1)
    curved, _ = apply_contrast_curve(clipped, "sigmoid", seed=seed)
    results["4_contrast_curved"] = np.clip(curved, 0, 1)
    
    # Step 3: AI artifacts (GPU-accelerated)
    with_chroma = add_chroma_noise(
        results["4_contrast_curved"], noise_strength=0.05, blur_sigma=20.0, seed=seed, device=device
    )
    results["5_chroma_noise"] = np.clip(with_chroma, 0, 1)
    
    with_sharpening = apply_unsharp_mask(results["5_chroma_noise"], radius=1.0, strength=1.5, device=device)
    results["6_unsharp_mask"] = np.clip(with_sharpening, 0, 1)
    
    # Step 4: 8-bit quantization
    quantized = quantize_to_8bit(results["6_unsharp_mask"], dither=False, seed=seed)
    results["7_quantized_8bit"] = quantized
    
    return results


# Apply degradation to multiple images and visualize
if aces_files:
    # Process first 2 images
    fig = plt.figure(figsize=(18, 12))
    
    for img_idx, aces_path in enumerate(aces_files[:2]):
        # Load and convert
        aces_image = load_aces_and_convert_srgb(aces_path, "sRGB")
        aces_clipped = np.clip(aces_image, 0, 1)
        
        # Apply degradation with deterministic seed
        degradated = apply_full_degradation(aces_clipped, seed=100 + img_idx, device=device)
        
        # Get RGB only
        original_rgb = degradated["original"][:, :, :3] if degradated["original"].shape[2] >= 3 else degradated["original"]
        
        # Plot progression
        stages = [
            ("Original", original_rgb),
            ("Color Warp", degradated.get("3_color_warped", original_rgb)[:, :, :3]),
            ("Contrast", degradated.get("4_contrast_curved", original_rgb)[:, :, :3]),
            ("Chroma Noise", degradated.get("5_chroma_noise", original_rgb)[:, :, :3]),
            ("Unsharp", degradated.get("6_unsharp_mask", original_rgb)[:, :, :3]),
            ("8-bit Quantized", degradated.get("7_quantized_8bit", original_rgb)[:, :, :3]),
        ]
        
        for stage_idx, (stage_name, stage_img) in enumerate(stages):
            ax = fig.add_subplot(2, 6, img_idx * 6 + stage_idx + 1)
            ax.imshow(stage_img)
            ax.set_title(stage_name, fontsize=10)
            ax.axis("off")
    
    plt.suptitle("Degradation Pipeline: ACES (Ground Truth) → 8-bit Synthetic Input", fontsize=14, y=0.99)
    plt.tight_layout()
    plt.savefig(output_dir / "06_full_degradation_pipeline.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("✓ Saved visualization: 06_full_degradation_pipeline.png")


## Section 7: Batch Process & Save Degraded Images

Export the ground-truth ACES and degraded 8-bit versions for use in training.
This creates paired (ground_truth, degraded_input) samples.

In [ ]:
import imageio

# Create output directories for training data
ground_truth_dir = output_dir / "ground_truth"
degraded_dir = output_dir / "degraded_input"
ground_truth_dir.mkdir(exist_ok=True)
degraded_dir.mkdir(exist_ok=True)

# Process all ACES images
print("\nBatch processing all ACES images...")
print(f"Total images: {len(aces_files)}")
print(f"Using device: {device}")

for idx, aces_path in enumerate(aces_files):
    try:
        # Load and convert ACES → sRGB
        aces_image = load_aces_and_convert_srgb(aces_path, "sRGB")
        aces_clipped = np.clip(aces_image, 0, 1)
        
        # Apply degradation with GPU acceleration
        degradated = apply_full_degradation(aces_clipped, seed=1000 + idx, device=device)
        
        # Get RGB channels
        gt_rgb = aces_clipped[:, :, :3]
        degraded_rgb = degradated.get("7_quantized_8bit", aces_clipped)[:, :, :3]
        
        # Convert to 8-bit for storage
        gt_8bit = (np.clip(gt_rgb, 0, 1) * 255).astype(np.uint8)
        degraded_8bit = (np.clip(degraded_rgb, 0, 1) * 255).astype(np.uint8)
        
        # Save
        base_name = aces_path.stem.replace("_aces", "")
        imageio.imwrite(ground_truth_dir / f"{base_name}_gt.png", gt_8bit)
        imageio.imwrite(degraded_dir / f"{base_name}_degraded.png", degraded_8bit)
        
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f"  [{idx+1}/{len(aces_files)}] {base_name}")
    
    except Exception as e:
        print(f"  ✗ Error processing {aces_path.name}: {e}")

print(f"\n✓ Saved {len(list(ground_truth_dir.glob('*.png')))} ground-truth images to {ground_truth_dir}")
print(f"✓ Saved {len(list(degraded_dir.glob('*.png')))} degraded images to {degraded_dir}")


## Summary

This notebook demonstrates the complete dataset synthesis pipeline for LuminaScale:

1. **ACES → sRGB Conversion**: Used OpenImageIO to convert cinema-grade ACES2065-1 images to display sRGB.
2. **Synthetic Degradation**: Applied realistic AI-artifact simulations:
   - Arbitrary color warping (random 3×3 matrix)
   - Stylized contrast curves (sigmoid/power)
   - Chroma noise (diffusion-like color blotches)
   - Unsharp masking (over-sharpening)
   - 8-bit quantization (introduces banding)

3. **Dataset Generation**: Created paired (ground_truth, degraded) samples for training.

### Key Insights

- **Banding is visible** after 8-bit quantization—especially in smooth gradients—making it the primary target for the BDE head.
- **Color warping** provides realistic color shift robustness for the ACES normalization head.
- **Artifacts compound**: The full pipeline creates convincingly "AI-generated" looking images.

### Next Steps

- Use ground_truth/ and degraded_input/ directories as training data
- Implement the two-stage cascade model (BDE → Color Normalization)
- Train on these synthetic pairs with perceptual + gradient losses